# RRC vs EIA Refinery Production Comparison

Compare Texas RRC refinery filings (`Products Manufactured`) against EIA
[Refinery Net Production](https://www.eia.gov/dnav/pet/pet_pnp_refp2_dc_nus_mbbl_m.htm),
aggregated by **refining district** from `reference/refinery_mapping.csv`.

We pull both `refp2` (refinery only) and `refp` (refinery + blender) so you can
see which EIA definition tracks RRC better.


In [231]:
# ── Config ─────────────────────────────────────────────────────────────────
from pathlib import Path

ROOT = Path('.').resolve()
RRC_CSV = ROOT / 'data' / 'all_production_20260511.csv'
MAPPING_CSV = ROOT / 'reference' / 'refinery_mapping.csv'
EIA_CACHE = ROOT / 'data' / 'eia_district_monthly.csv'
EIA_PRODUCT_CACHE = ROOT / 'data' / 'eia_district_products_monthly.csv'

DATE_FROM = '2020-01'
DATE_TO = '2025-12'

# EIA refining-district codes (PADD 3 sub-districts)
EIA_DISTRICT_MAP = {
    'R3B': 'Texas Gulf Coast',
    'R3A': 'Texas Inland',
}

# RRC material → EIA product code (refp2/refp routes)
PRODUCT_MAP = {
    'Motor Gasoline': 'EPM0F',
    'Diesel Fuel': 'EPD0',
    'Kerosene-Type Jet Fuel': 'EPJK',
    'Kerosene': 'EPPK',
    'Aviation Gasoline': 'EPPV',
    'Propane': 'EPLLPA',
    'Butane': 'EPLLBAN',
    'Petrochemical Feedstock': 'EPPP0',
    'Residual Fuel Oil': 'EPPR',
    'Lubricants': 'EPPL',
    'Special Naphthas': 'EPPNS',
    'Asphalt': 'EPPA',
}

import warnings
warnings.filterwarnings('ignore')

import os
import requests
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from dotenv import load_dotenv

load_dotenv(ROOT / '.env')
EIA_API_KEY = os.getenv('EIA_API_KEY', '').strip().strip("'").strip('"')
assert EIA_API_KEY, 'Set EIA_API_KEY in .env'


In [232]:
# ── Load RRC production + refinery mapping ─────────────────────────────────
mapping = pd.read_csv(MAPPING_CSV, dtype={'id': str})
mapping = mapping.rename(columns={'refining district': 'district'})

rrc = pd.read_csv(RRC_CSV, dtype={'id': str})
rrc['date'] = pd.to_datetime(rrc['date'], dayfirst=True)
rrc['period'] = rrc['date'].dt.to_period('M').astype(str)
rrc['balanced'] = rrc['balanced'].astype(str).str.upper().map({'TRUE': True, 'FALSE': False})

for col in ['Products Manufactured', 'Input', 'Fuel Used', 'Receipts', 'Deliveries']:
    rrc[col] = pd.to_numeric(rrc[col], errors='coerce').fillna(0)

# RRC CSV already has plant_name/ownser from extraction — only add district label
rrc = rrc.merge(mapping[['id', 'district']], on='id', how='left')

if DATE_FROM:
    rrc = rrc[rrc['period'] >= DATE_FROM]
if DATE_TO:
    rrc = rrc[rrc['period'] <= DATE_TO]

print(f'RRC rows: {len(rrc):,}')
print(f'Refineries in data: {rrc["id"].nunique()} | Districts: {sorted(rrc["district"].dropna().unique())}')
print(f'Period: {rrc["period"].min()} → {rrc["period"].max()}')


RRC rows: 46,226
Refineries in data: 28 | Districts: ['Texas Gulf Coast', 'Texas Inland']
Period: 2020-01 → 2025-12


In [233]:
# ── RRC district aggregates ────────────────────────────────────────────────
# Products Manufactured excludes the Crude Oil balance row; that row tracks inputs.
products = rrc[rrc['material'] != 'Crude Oil'].copy()

rrc_prod = (
    products.groupby(['district', 'period'], as_index=False)['Products Manufactured']
    .sum()
    .rename(columns={'Products Manufactured': 'rrc_mbbl'})
)
rrc_prod['rrc_mbbl'] /= 1000  # barrels -> thousand barrels

# Some refineries (e.g. Galena Park) book crude charge in Fuel Used, not Input.
crude_charge = rrc[rrc['material'] == 'Crude Oil'].copy()
crude_charge['crude_charge_bbl'] = crude_charge['Input'] + crude_charge['Fuel Used']

rrc_crude_in = (
    crude_charge
    .groupby(['district', 'period'], as_index=False)['crude_charge_bbl']
    .sum()
    .rename(columns={'crude_charge_bbl': 'rrc_crude_input_mbbl'})
)
rrc_crude_in['rrc_crude_input_mbbl'] /= 1000

# Per-refinery contribution (for drill-down)
rrc_by_refinery = (
    products.groupby(['district', 'id', 'plant_name', 'period'], as_index=False)['Products Manufactured']
    .sum()
)
rrc_by_refinery['mbbl'] = rrc_by_refinery['Products Manufactured'] / 1000

rrc_prod.head()


,district,period,rrc_mbbl
0,Texas Gulf Coast,2020-01,164356.332
1,Texas Gulf Coast,2020-02,129718.194
2,Texas Gulf Coast,2020-03,155182.742
3,Texas Gulf Coast,2020-04,122600.018
4,Texas Gulf Coast,2020-05,125188.113


In [234]:
# ── Fetch EIA data (cached) ─────────────────────────────────────────────────
def fetch_eia_route(route: str, product: str, start: str, end: str) -> pd.DataFrame:
    """Pull monthly MBBL series for Texas refining districts."""
    url = f'https://api.eia.gov/v2/petroleum/pnp/{route}/data/'
    params = {
        'api_key': EIA_API_KEY,
        'frequency': 'monthly',
        'data[0]': 'value',
        'facets[duoarea][]': list(EIA_DISTRICT_MAP),
        'facets[product][]': [product],
        'start': start,
        'end': end,
        'sort[0][column]': 'period',
        'sort[0][direction]': 'asc',
        'length': 5000,
    }
    rows = []
    offset = 0
    while True:
        params['offset'] = offset
        resp = requests.get(url, params=params, timeout=60)
        resp.raise_for_status()
        batch = resp.json()['response']['data']
        if not batch:
            break
        rows.extend(batch)
        if len(batch) < 5000:
            break
        offset += 5000

    df = pd.DataFrame(rows)
    df = df[df['units'] == 'MBBL'].copy()
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    df['district'] = df['duoarea'].map(EIA_DISTRICT_MAP)
    df['route'] = route
    df['product_code'] = product
    return df[['route', 'district', 'period', 'product_code', 'product-name', 'value', 'series']]


if EIA_CACHE.exists():
    eia_all = pd.read_csv(EIA_CACHE)
    print(f'Loaded cached EIA data: {EIA_CACHE}')
else:
    frames = []
    for route in ['refp2', 'refp']:
        frames.append(fetch_eia_route(route, 'EP00', DATE_FROM, DATE_TO))
    for route in ['inpt2', 'inpt']:
        frames.append(fetch_eia_route(route, 'EPC0', DATE_FROM, DATE_TO))
    eia_all = pd.concat(frames, ignore_index=True)
    eia_all.to_csv(EIA_CACHE, index=False)
    print(f'Fetched and cached EIA data → {EIA_CACHE}')

eia_all['metric'] = eia_all['route'].map({
    'refp2': 'production_refinery',
    'refp': 'production_refinery_blender',
    'inpt2': 'crude_input_refinery',
    'inpt': 'crude_input_refinery_blender',
})

eia_all.head()


Loaded cached EIA data: C:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\rrc_texas_info\data\eia_district_monthly.csv


,route,district,period,product_code,product-name,value,series,metric
0,refp2,Texas Gulf Coast,2020-01,EP00,Crude Oil and Petroleum Products,103536,MTTRX_R3B_1,production_refinery
1,refp2,Texas Inland,2020-01,EP00,Crude Oil and Petroleum Products,11043,MTTRX_R3A_1,production_refinery
2,refp2,Texas Inland,2020-02,EP00,Crude Oil and Petroleum Products,10103,MTTRX_R3A_1,production_refinery
3,refp2,Texas Gulf Coast,2020-02,EP00,Crude Oil and Petroleum Products,95547,MTTRX_R3B_1,production_refinery
4,refp2,Texas Inland,2020-03,EP00,Crude Oil and Petroleum Products,11940,MTTRX_R3A_1,production_refinery


In [235]:
eia_all.to_csv('total_info.csv')

In [236]:
# ── Merge RRC vs EIA (production totals) ───────────────────────────────────
eia_prod = eia_all[eia_all['metric'].str.startswith('production')].rename(columns={'value': 'eia_mbbl'})

compare = rrc_prod.merge(
    eia_prod[['district', 'period', 'route', 'metric', 'eia_mbbl']],
    on=['district', 'period'],
    how='inner',
)

compare['diff_mbbl'] = compare['rrc_mbbl'] - compare['eia_mbbl']
compare['pct_diff'] = np.where(compare['eia_mbbl'] > 0, 100 * compare['diff_mbbl'] / compare['eia_mbbl'], np.nan)

# Summary: which EIA route fits RRC better?
fit = (
    compare.groupby(['district', 'route'])
    .agg(
        n_months=('period', 'count'),
        mean_pct_diff=('pct_diff', 'mean'),
        median_pct_diff=('pct_diff', 'median'),
        mae_mbbl=('diff_mbbl', lambda s: np.abs(s).mean()),
        corr=('rrc_mbbl', lambda s: s.corr(compare.loc[s.index, 'eia_mbbl'])),
    )
    .reset_index()
    .sort_values(['district', 'mae_mbbl'])
)
print('Fit summary (lower MAE / median % diff = better):')
display(fit)


Fit summary (lower MAE / median % diff = better):


,district,route,n_months,mean_pct_diff,median_pct_diff,mae_mbbl,corr
0,Texas Gulf Coast,refp,72,46.725307,47.201615,51291.457385,0.933077
1,Texas Gulf Coast,refp2,72,62.593966,62.874707,61932.790719,0.928066
3,Texas Inland,refp2,72,68.383477,68.111351,8648.460252,0.896137
2,Texas Inland,refp,72,-30.413270,-30.842900,9343.706415,0.775712


In [237]:
# ── Time series: RRC vs refp2 vs refp ───────────────────────────────────────
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, subplot_titles=list(EIA_DISTRICT_MAP.values()))

colors = {'rrc_mbbl': '#1f77b4', 'refp2': '#ff7f0e', 'refp': '#2ca02c'}

for i, district in enumerate(EIA_DISTRICT_MAP.values(), start=1):
    rrc_line = rrc_prod[rrc_prod['district'] == district]
    fig.add_trace(
        go.Scatter(x=rrc_line['period'], y=rrc_line['rrc_mbbl'], mode='lines', name='RRC', line=dict(color=colors['rrc_mbbl']), legendgroup='rrc', showlegend=(i == 1)),
        row=i, col=1,
    )
    for route in ['refp2', 'refp']:
        eia_line = eia_prod[(eia_prod['district'] == district) & (eia_prod['route'] == route)]
        fig.add_trace(
            go.Scatter(x=eia_line['period'], y=eia_line['eia_mbbl'], mode='lines', name=route, line=dict(color=colors[route], dash='dot' if route == 'refp' else 'solid'), legendgroup=route, showlegend=(i == 1)),
            row=i, col=1,
        )

fig.update_layout(height=700, title='District production: RRC vs EIA refp2 / refp (EP00, mbbl)', hovermode='x unified')
fig.update_yaxes(title_text='mbbl')
fig.show()


In [238]:
# ── % difference over time ───────────────────────────────────────────────────
fig = px.line(
    compare,
    x='period', y='pct_diff', color='route', facet_row='district',
    title='% difference: (RRC − EIA) / EIA by district and EIA route',
    labels={'pct_diff': '% diff', 'period': ''},
)
fig.add_hline(y=0, line_dash='dash', line_color='gray')
fig.update_layout(height=500, hovermode='x unified')
fig.show()


In [239]:
# ── Scatter: RRC vs EIA with 1:1 line ───────────────────────────────────────
fig = make_subplots(cols=2, subplot_titles=list(EIA_DISTRICT_MAP.values()))
max_val = max(compare['eia_mbbl'].max(), compare['rrc_mbbl'].max())

for col, district in enumerate(EIA_DISTRICT_MAP.values(), start=1):
    sub = compare[compare['district'] == district]
    for route in ['refp2', 'refp']:
        s = sub[sub['route'] == route]
        fig.add_trace(
            go.Scatter(x=s['eia_mbbl'], y=s['rrc_mbbl'], mode='markers', name=route, legendgroup=route, showlegend=(col == 1), opacity=0.6),
            row=1, col=col,
        )
    fig.add_trace(
        go.Scatter(x=[0, max_val], y=[0, max_val], mode='lines', name='1:1', line=dict(dash='dash', color='black'), showlegend=(col == 1)),
        row=1, col=col,
    )

fig.update_layout(height=400, title='RRC vs EIA monthly production (mbbl)')
fig.update_xaxes(title_text='EIA mbbl')
fig.update_yaxes(title_text='RRC mbbl')
fig.show()


In [240]:
# ── Product-level comparison (mapped products only) ──────────────────────────
mapped_materials = [m for m in PRODUCT_MAP if m in products['material'].unique()]
rrc_by_product = (
    products[products['material'].isin(mapped_materials)]
    .assign(eia_product=lambda d: d['material'].map(PRODUCT_MAP))
    .groupby(['district', 'period', 'material', 'eia_product'], as_index=False)['Products Manufactured']
    .sum()
)
rrc_by_product['rrc_mbbl'] = rrc_by_product['Products Manufactured'] / 1000

# Product-level EIA (refp2), cached after first fetch
if EIA_PRODUCT_CACHE.exists():
    eia_products = pd.read_csv(EIA_PRODUCT_CACHE)
    print(f'Loaded cached product EIA: {EIA_PRODUCT_CACHE}')
else:
    eia_product_frames = []
    for pcode in sorted(rrc_by_product['eia_product'].unique()):
        eia_product_frames.append(fetch_eia_route('refp2', pcode, DATE_FROM, DATE_TO))
    eia_products = pd.concat(eia_product_frames, ignore_index=True)
    eia_products.to_csv(EIA_PRODUCT_CACHE, index=False)
    print(f'Fetched product EIA → {EIA_PRODUCT_CACHE}')
eia_products = eia_products.rename(columns={'value': 'eia_mbbl', 'product-name': 'eia_product_name'})

prod_compare = rrc_by_product.merge(
    eia_products[['district', 'period', 'product_code', 'eia_mbbl', 'eia_product_name']],
    left_on=['district', 'period', 'eia_product'],
    right_on=['district', 'period', 'product_code'],
    how='inner',
)
prod_compare['pct_diff'] = np.where(prod_compare['eia_mbbl'] > 0, 100 * (prod_compare['rrc_mbbl'] - prod_compare['eia_mbbl']) / prod_compare['eia_mbbl'], np.nan)

prod_fit = (
    prod_compare.groupby(['district', 'material'])
    .agg(median_pct_diff=('pct_diff', 'median'), mae_mbbl=('rrc_mbbl', lambda s: (s - prod_compare.loc[s.index, 'eia_mbbl']).abs().mean()))
    .reset_index()
    .sort_values('mae_mbbl')
)
print('Product-level fit (refp2):')
prod_fit


Loaded cached product EIA: C:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\rrc_texas_info\data\eia_district_products_monthly.csv
Product-level fit (refp2):


,district,material,median_pct_diff,mae_mbbl
15,Texas Inland,Lubricants,3.536000,1.677500
10,Texas Inland,Aviation Gasoline,-0.033295,2.146181
13,Texas Inland,Kerosene,-1.678283,4.225097
14,Texas Inland,Kerosene-Type Jet Fuel,-0.009303,24.264470
19,Texas Inland,Special Naphthas,-100.000000,56.125319
12,Texas Inland,Diesel Fuel,0.865246,73.301708
0,Texas Gulf Coast,Aviation Gasoline,0.005417,100.215000
11,Texas Inland,Butane,-26.390010,110.285028
17,Texas Inland,Petrochemical Feedstock,77.135881,128.980389
5,Texas Gulf Coast,Lubricants,11.059130,206.915819


In [241]:
# Top product gaps — where definitional / coverage issues show up
latest = prod_compare['period'].max()
prod_latest = prod_compare[prod_compare['period'] == latest].copy()
prod_latest['diff_mbbl'] = prod_latest['rrc_mbbl'] - prod_latest['eia_mbbl']
prod_latest.sort_values('diff_mbbl', key=abs, ascending=False).head(12)


,district,period,material,eia_product,Products Manufactured,rrc_mbbl,product_code,eia_mbbl,eia_product_name,pct_diff,diff_mbbl
708,Texas Gulf Coast,2025-12,Motor Gasoline,EPM0F,75490789.0,75490.789,EPM0F,9949,Finished Motor Gasoline,658.777656,65541.789
1428,Texas Inland,2025-12,Motor Gasoline,EPM0F,13129369.0,13129.369,EPM0F,2572,Finished Motor Gasoline,410.473134,10557.369
704,Texas Gulf Coast,2025-12,Diesel Fuel,EPD0,39048072.0,39048.072,EPD0,47959,Distillate Fuel Oil,-18.580304,-8910.928
706,Texas Gulf Coast,2025-12,Kerosene-Type Jet Fuel,EPJK,11698255.0,11698.255,EPJK,18191,Kerosene-Type Jet Fuel,-35.692073,-6492.745
705,Texas Gulf Coast,2025-12,Kerosene,EPPK,3160993.0,3160.993,EPPK,92,Kerosene,3335.861957,3068.993
710,Texas Gulf Coast,2025-12,Propane,EPLLPA,4021405.0,4021.405,EPLLPA,2570,Propane,56.474903,1451.405
709,Texas Gulf Coast,2025-12,Petrochemical Feedstock,EPPP0,3784059.0,3784.059,EPPP0,4778,Petrochemical Feedstocks,-20.802449,-993.941
702,Texas Gulf Coast,2025-12,Aviation Gasoline,EPPV,776936.0,776.936,EPPV,146,Aviation Gasoline,432.147945,630.936
1429,Texas Inland,2025-12,Petrochemical Feedstock,EPPP0,-125121.0,-125.121,EPPP0,164,Petrochemical Feedstocks,-176.293293,-289.121
1430,Texas Inland,2025-12,Propane,EPLLPA,571249.0,571.249,EPLLPA,310,Propane,84.273871,261.249


## Product explorer

Drill into **individual products** to see which RRC lines drive the district-level gap vs EIA.
Use the dropdown charts below to switch product and district; the decomposition chart shows
how much each mapped product contributes to the total mbbl difference.

In [242]:
# ── Product-level EIA: refp2 + refp (extend cache if needed) ─────────────────
EIA_PRODUCT_CACHE_BOTH = ROOT / 'data' / 'eia_district_products_monthly_both.csv'

product_codes = sorted(rrc_by_product['eia_product'].unique())

if EIA_PRODUCT_CACHE_BOTH.exists():
    eia_products_all = pd.read_csv(EIA_PRODUCT_CACHE_BOTH)
    print(f'Loaded {EIA_PRODUCT_CACHE_BOTH}')
else:
    frames = []
    for route in ['refp2', 'refp']:
        for pcode in product_codes:
            frames.append(fetch_eia_route(route, pcode, DATE_FROM, DATE_TO))
    eia_products_all = pd.concat(frames, ignore_index=True)
    eia_products_all.to_csv(EIA_PRODUCT_CACHE_BOTH, index=False)
    print(f'Cached refp2 + refp product EIA -> {EIA_PRODUCT_CACHE_BOTH}')

eia_products_all = eia_products_all.rename(columns={'value': 'eia_mbbl', 'product-name': 'eia_product_name'})
eia_products_all['eia_mbbl'] = pd.to_numeric(eia_products_all['eia_mbbl'], errors='coerce')

prod_compare_all = rrc_by_product.merge(
    eia_products_all[['route', 'district', 'period', 'product_code', 'eia_mbbl', 'eia_product_name']],
    left_on=['district', 'period', 'eia_product'],
    right_on=['district', 'period', 'product_code'],
    how='inner',
)
prod_compare_all['diff_mbbl'] = prod_compare_all['rrc_mbbl'] - prod_compare_all['eia_mbbl']
prod_compare_all['pct_diff'] = np.where(
    prod_compare_all['eia_mbbl'] > 0,
    100 * prod_compare_all['diff_mbbl'] / prod_compare_all['eia_mbbl'],
    np.nan,
)

# RRC products with no EIA mapping — often explain part of the total gap
unmapped_materials = sorted(set(products['material'].unique()) - set(PRODUCT_MAP))
rrc_unmapped = (
    products[products['material'].isin(unmapped_materials)]
    .groupby(['district', 'period', 'material'], as_index=False)['Products Manufactured']
    .sum()
)
rrc_unmapped['rrc_mbbl'] = rrc_unmapped['Products Manufactured'] / 1000

print(f'Mapped products: {len(PRODUCT_MAP)} | Unmapped RRC materials: {len(unmapped_materials)}')
print('Unmapped:', unmapped_materials)
prod_compare_all.groupby(['route', 'district', 'material']).size().head(8)

Loaded C:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\rrc_texas_info\data\eia_district_products_monthly_both.csv
Mapped products: 12 | Unmapped RRC materials: 13
Unmapped: ['Bunker C', 'Butane-Propane', 'Fuel Oils #4 For Non-Utility Use', 'Fuel Oils #4 For Utility Use', 'Fuel Oils #5,#6 For Non-Utility Use', 'Fuel Oils #5,#6 For Utility Use', 'Home Heating Oil', 'Miscellaneous', 'Naphtha-Type Jet Fuel', 'Navy Special', 'Other Middle Distillates', 'Other Residual Fuels', 'Solvent Products']


route  district          material               
refp   Texas Gulf Coast  Aviation Gasoline          71
                         Butane                     72
                         Diesel Fuel                72
                         Kerosene                   70
                         Kerosene-Type Jet Fuel     72
                         Lubricants                 72
                         Motor Gasoline             72
                         Petrochemical Feedstock    72
dtype: int64

In [243]:
# ── Heatmap: median % diff by product × district (refp2) ─────────────────────
prod_fit_refp2 = (
    prod_compare_all[prod_compare_all['route'] == 'refp2']
    .groupby(['district', 'material'], as_index=False)
    .agg(
        median_pct_diff=('pct_diff', 'median'),
        median_diff_mbbl=('diff_mbbl', 'median'),
        mae_mbbl=('diff_mbbl', lambda s: s.abs().mean()),
    )
)

heatmap_pct = prod_fit_refp2.pivot(index='material', columns='district', values='median_pct_diff')
heatmap_mae = prod_fit_refp2.pivot(index='material', columns='district', values='mae_mbbl')

fig = make_subplots(rows=1, cols=2, subplot_titles=['Median % diff (RRC−EIA)/EIA', 'Mean |diff| (mbbl)'])
fig.add_trace(go.Heatmap(z=heatmap_pct.values, x=heatmap_pct.columns, y=heatmap_pct.index, colorscale='RdBu', zmid=0, colorbar=dict(x=0.45, title='%')), row=1, col=1)
fig.add_trace(go.Heatmap(z=heatmap_mae.values, x=heatmap_mae.columns, y=heatmap_mae.index, colorscale='YlOrRd', colorbar=dict(x=1.02, title='mbbl')), row=1, col=2)
fig.update_layout(height=450, width=950, title='Product-level fit heatmap (EIA refp2)')
fig.show()

In [244]:
# ── Interactive product charts (Plotly dropdowns) ─────────────────────────────
def _product_dropdown_fig(df, y_col, title, y_label):
    """Build a line chart with dropdowns for product and EIA route."""
    materials = sorted(df['material'].unique())
    routes = sorted(df['route'].unique())
    districts = sorted(df['district'].unique())

    fig = go.Figure()
    trace_meta = []  # (material, route, district, trace_idx)

    for mat in materials:
        for route in routes:
            for district in districts:
                sub = df[(df['material'] == mat) & (df['route'] == route) & (df['district'] == district)].sort_values('period')
                if sub.empty:
                    continue
                fig.add_trace(go.Scatter(
                    x=sub['period'], y=sub[y_col], mode='lines+markers', name=f'{mat} | {district} | {route}',
                    visible=False,
                ))
                trace_meta.append((mat, route, district, len(fig.data) - 1))

    # Default: Motor Gasoline if present, else first material; refp2; Gulf Coast
    default = ('Motor Gasoline', 'refp2', 'Texas Gulf Coast')
    if not any(m == default[0] for m, _, _, _ in trace_meta):
        default = trace_meta[0][:3]

    for mat, route, district, idx in trace_meta:
        fig.data[idx].visible = (mat, route, district) == default

    product_buttons, route_buttons, district_buttons = [], [], []
    n = len(fig.data)

    for mat in materials:
        visible = [fig.data[i].visible if (trace_meta[i][0] == mat and trace_meta[i][1] == default[1] and trace_meta[i][2] == default[2]) else False for i in range(n)]
        if not any(visible):
            visible = [trace_meta[i][0] == mat and trace_meta[i][1] == routes[0] and trace_meta[i][2] == districts[0] for i in range(n)]
        product_buttons.append(dict(label=mat, method='update', args=[{'visible': visible}, {'title': f'{title} — {mat}'}]))

    for route in routes:
        visible = [trace_meta[i][0] == default[0] and trace_meta[i][1] == route and trace_meta[i][2] == default[2] for i in range(n)]
        route_buttons.append(dict(label=route, method='update', args=[{'visible': visible}, {'title': f'{title} — {default[0]} ({route})'}]))

    for district in districts:
        visible = [trace_meta[i][0] == default[0] and trace_meta[i][1] == default[1] and trace_meta[i][2] == district for i in range(n)]
        district_buttons.append(dict(label=district, method='update', args=[{'visible': visible}, {'title': f'{title} — {default[0]} | {district}'}]))

    # Combined buttons: product × route × district
    combo_buttons = []
    for mat in materials:
        for route in routes:
            for district in districts:
                visible = [trace_meta[i] == (mat, route, district) for i in range(n)]
                if any(visible):
                    combo_buttons.append(dict(
                        label=f'{mat} | {district} | {route}',
                        method='update',
                        args=[{'visible': visible}, {'title': f'{title} — {mat} | {district} | {route}'}],
                    ))

    fig.update_layout(
        title=f'{title} — {default[0]} | {default[2]} | {default[1]}',
        xaxis_title='period',
        yaxis_title=y_label,
        hovermode='x unified',
        height=450,
        updatemenus=[{
            'buttons': combo_buttons,
            'direction': 'down',
            'showactive': True,
            'x': 0.0,
            'y': 1.15,
            'xanchor': 'left',
            'yanchor': 'top',
        }],
    )
    return fig


# Side-by-side RRC vs EIA level (two traces per selection)
def product_level_dropdown(df):
    fig = go.Figure()
    combos = []

    for mat in sorted(df['material'].unique()):
        for route in sorted(df['route'].unique()):
            for district in sorted(df['district'].unique()):
                sub = df[(df['material'] == mat) & (df['route'] == route) & (df['district'] == district)].sort_values('period')
                if sub.empty:
                    continue
                combos.append((mat, route, district))
                fig.add_trace(go.Scatter(x=sub['period'], y=sub['rrc_mbbl'], name='RRC', line=dict(color='#1f77b4'), visible=False))
                fig.add_trace(go.Scatter(x=sub['period'], y=sub['eia_mbbl'], name='EIA', line=dict(color='#ff7f0e', dash='dot'), visible=False))

    default_idx = next((i for i, c in enumerate(combos) if c == ('Motor Gasoline', 'refp2', 'Texas Gulf Coast')), 0)
    fig.data[2 * default_idx].visible = True
    fig.data[2 * default_idx + 1].visible = True
    default = combos[default_idx]

    buttons = []
    for i, (mat, route, district) in enumerate(combos):
        visible = [False] * (2 * len(combos))
        visible[2 * i] = True
        visible[2 * i + 1] = True
        buttons.append(dict(
            label=f'{mat} | {district} | {route}',
            method='update',
            args=[{'visible': visible}, {'title': f'RRC vs EIA production — {mat} | {district} | {route}'}],
        ))

    fig.update_layout(
        title=f'RRC vs EIA production — {default[0]} | {default[2]} | {default[1]}',
        yaxis_title='mbbl',
        hovermode='x unified',
        height=450,
        updatemenus=[{'buttons': buttons, 'direction': 'down', 'showactive': True, 'x': 0, 'y': 1.12}],
    )
    return fig

fig_levels = product_level_dropdown(prod_compare_all)
fig_levels.show()

fig_pct = _product_dropdown_fig(prod_compare_all, 'pct_diff', 'RRC vs EIA % difference', '% diff')
fig_pct.add_hline(y=0, line_dash='dash', line_color='gray')
fig_pct.show()

In [245]:
# ── Gap decomposition: which products drive the district difference? ─────────
# Compare mapped-product diffs vs total EP00 gap; unmapped RRC fills the remainder.

def gap_decomposition(period, route='refp2'):
    total = compare[compare['route'] == route].copy()
    total = total[total['period'] == period][['district', 'rrc_mbbl', 'eia_mbbl', 'diff_mbbl']].rename(
        columns={'diff_mbbl': 'total_diff_mbbl'}
    )

    prod = prod_compare_all[(prod_compare_all['route'] == route) & (prod_compare_all['period'] == period)]
    prod_gap = (
        prod.groupby(['district', 'material'], as_index=False)['diff_mbbl']
        .sum()
        .rename(columns={'diff_mbbl': 'mapped_diff_mbbl'})
    )

    unmapped = rrc_unmapped[rrc_unmapped['period'] == period].groupby('district', as_index=False)['rrc_mbbl'].sum()
    unmapped = unmapped.rename(columns={'rrc_mbbl': 'unmapped_rrc_mbbl'})

    summary = total.merge(
        prod_gap.groupby('district', as_index=False)['mapped_diff_mbbl'].sum(),
        on='district', how='left',
    ).merge(unmapped, on='district', how='left')
    summary['unmapped_rrc_mbbl'] = summary['unmapped_rrc_mbbl'].fillna(0)
    summary['mapped_diff_mbbl'] = summary['mapped_diff_mbbl'].fillna(0)
    summary['unexplained_mbbl'] = summary['total_diff_mbbl'] - summary['mapped_diff_mbbl']

    return prod_gap, summary

decomp_period = prod_compare_all['period'].max()
prod_gap, gap_summary = gap_decomposition(decomp_period, route='refp2')
print(f'Gap decomposition for {decomp_period} (refp2):')
display(gap_summary)

# Stacked bar: mapped product contributions to diff (positive = RRC > EIA)
fig = px.bar(
    prod_gap,
    x='district', y='mapped_diff_mbbl', color='material',
    title=f'Mapped product gap vs EIA by district ({decomp_period}, mbbl)',
    labels={'mapped_diff_mbbl': 'RRC − EIA (mbbl)', 'material': 'product'},
    barmode='relative',
)
fig.add_hline(y=0, line_dash='dash', line_color='gray')
fig.update_layout(height=500)
fig.show()

# Unmapped RRC production (no EIA counterpart in PRODUCT_MAP)
unmapped_latest = rrc_unmapped[rrc_unmapped['period'] == decomp_period].copy()
fig2 = px.bar(
    unmapped_latest.sort_values('rrc_mbbl', ascending=False),
    x='material', y='rrc_mbbl', color='district', facet_col='district',
    title=f'Unmapped RRC production only ({decomp_period}, mbbl) — not in EIA comparison',
    labels={'rrc_mbbl': 'RRC mbbl'},
)
fig2.update_layout(height=450, showlegend=False)
fig2.show()

Gap decomposition for 2025-12 (refp2):


,district,rrc_mbbl,eia_mbbl,total_diff_mbbl,mapped_diff_mbbl,unmapped_rrc_mbbl,unexplained_mbbl
0,Texas Gulf Coast,182219.460,104268,77951.460,54759.271,40689.189,23192.189
1,Texas Inland,24124.898,13985,10139.898,10654.868,1596.030,-514.970


In [246]:
# ── refp2 vs refp at product level: which route tracks RRC better? ───────────
prod_route_fit = (
    prod_compare_all.groupby(['route', 'district', 'material'])
    .agg(
        median_pct_diff=('pct_diff', 'median'),
        mae_mbbl=('diff_mbbl', lambda s: s.abs().mean()),
    )
    .reset_index()
)

materials_sorted = sorted(prod_compare_all['material'].unique())
fig = px.bar(
    prod_route_fit,
    x='material', y='mae_mbbl', color='route', facet_col='district',
    barmode='group',
    title='Product-level MAE: RRC vs EIA by route (refp2 vs refp)',
    labels={'mae_mbbl': 'mean |RRC−EIA| (mbbl)'},
    category_orders={'material': materials_sorted},
)
fig.update_layout(height=550, xaxis_tickangle=-45)
fig.show()

# Time-series overlay: refp2 vs refp for a single product (configurable)
FOCUS_PRODUCT = 'Motor Gasoline'
FOCUS_DISTRICT = 'Texas Gulf Coast'

focus = prod_compare_all[
    (prod_compare_all['material'] == FOCUS_PRODUCT) & (prod_compare_all['district'] == FOCUS_DISTRICT)
]
rrc_line = focus.drop_duplicates('period').sort_values('period')
fig = go.Figure()
fig.add_trace(go.Scatter(x=rrc_line['period'], y=rrc_line['rrc_mbbl'], name='RRC', line=dict(color='#1f77b4')))
for route in ['refp2', 'refp']:
    sub = focus[focus['route'] == route].sort_values('period')
    fig.add_trace(go.Scatter(x=sub['period'], y=sub['eia_mbbl'], name=f'EIA {route}', line=dict(dash='dot')))
fig.update_layout(
    title=f'{FOCUS_PRODUCT} | {FOCUS_DISTRICT} — RRC vs EIA routes',
    yaxis_title='mbbl', hovermode='x unified', height=420,
)
fig.show()
print('Change FOCUS_PRODUCT / FOCUS_DISTRICT above to inspect other products.')

Change FOCUS_PRODUCT / FOCUS_DISTRICT above to inspect other products.


In [247]:
# ── Crude input: RRC vs EIA inpt2 / inpt ─────────────────────────────────────
eia_input = eia_all[eia_all['metric'].str.startswith('crude_input')].rename(columns={'value': 'eia_crude_mbbl'})

input_compare = rrc_crude_in.merge(
    eia_input[['district', 'period', 'route', 'eia_crude_mbbl']],
    on=['district', 'period'],
    how='inner',
)
input_compare['pct_diff'] = np.where(
    input_compare['eia_crude_mbbl'] > 0,
    100 * (input_compare['rrc_crude_input_mbbl'] - input_compare['eia_crude_mbbl']) / input_compare['eia_crude_mbbl'],
    np.nan,
)

input_fit = (
    input_compare.groupby(['district', 'route'])
    .agg(median_pct_diff=('pct_diff', 'median'), mae_mbbl=('rrc_crude_input_mbbl', lambda s: (s - input_compare.loc[s.index, 'eia_crude_mbbl']).abs().mean()))
    .reset_index()
    .sort_values(['district', 'mae_mbbl'])
)
print('Crude input fit:')
display(input_fit)

fig = make_subplots(rows=2, cols=1, subplot_titles=list(EIA_DISTRICT_MAP.values()))
for i, district in enumerate(EIA_DISTRICT_MAP.values(), start=1):
    sub_r = rrc_crude_in[rrc_crude_in['district'] == district]
    fig.add_trace(go.Scatter(x=sub_r['period'], y=sub_r['rrc_crude_input_mbbl'], name='RRC crude input', legendgroup='rrc', showlegend=(i==1)), row=i, col=1)
    for route in ['inpt2', 'inpt']:
        sub_e = eia_input[(eia_input['district'] == district) & (eia_input['route'] == route)]
        fig.add_trace(go.Scatter(x=sub_e['period'], y=sub_e['eia_crude_mbbl'], name=route, legendgroup=route, showlegend=(i==1), line=dict(dash='dot' if route=='inpt' else 'solid')), row=i, col=1)
fig.update_layout(height=600, title='Crude input by district (mbbl)')
fig.show()


Crude input fit:


,district,route,median_pct_diff,mae_mbbl
0,Texas Gulf Coast,inpt,-1.197781,2966.259517
1,Texas Gulf Coast,inpt2,-1.197781,2966.259517
2,Texas Inland,inpt,0.053526,73.986319
3,Texas Inland,inpt2,0.053526,73.986319


## Per-refinery crude intake (kbd)

Monthly **crude charge** from each refinery's RRC `Crude Oil` balance row, converted to
**thousand barrels per day** (kbd = monthly `Input` + `Fuel Used` ÷ days in month ÷ 1,000).
Some plants (e.g. Galena Park) report crude intake in `Fuel Used` rather than `Input`.

Use the refinery dropdown to pick a plant. A dashed **period-average** line can be toggled
on/off with the second dropdown. Mapped plants with no usable crude filings are listed below the table.

In [248]:
# ── Per-refinery crude intake (kbd) ───────────────────────────────────────────
crude_rows = rrc[rrc['material'] == 'Crude Oil'].copy()
crude_rows['crude_charge_bbl'] = crude_rows['Input'] + crude_rows['Fuel Used']

rrc_crude_by_refinery = (
    crude_rows.groupby(['district', 'id', 'plant_name', 'period', 'iir_map'], as_index=False)['crude_charge_bbl']
    .sum()
    .rename(columns={'crude_charge_bbl': 'input_bbl'})
)
rrc_crude_by_refinery['period_dt'] = pd.to_datetime(rrc_crude_by_refinery['period'] + '-01')
# kbd normalizes for different month lengths (EIA-style daily rate).
rrc_crude_by_refinery['kbd'] = (
    rrc_crude_by_refinery['input_bbl']
    / rrc_crude_by_refinery['period_dt'].dt.days_in_month
    / 1000
)

ids_with_crude = set(crude_rows['id'].unique())
ids_in_rrc = set(rrc['id'].unique())
crude_input_totals = crude_rows.groupby('id')['crude_charge_bbl'].sum()

refinery_coverage = (
    mapping.rename(columns={'refining district': 'district'})
    [['id', 'ownser', 'plant_name', 'district']]
    .drop_duplicates('id')
    .copy()
)
refinery_coverage['in_rrc_extract'] = refinery_coverage['id'].isin(ids_in_rrc)
refinery_coverage['has_crude_row'] = refinery_coverage['id'].isin(ids_with_crude)
refinery_coverage['crude_input_total_bbl'] = refinery_coverage['id'].map(crude_input_totals).fillna(0)
refinery_coverage['status'] = np.select(
    [
        ~refinery_coverage['in_rrc_extract'],
        ~refinery_coverage['has_crude_row'],
        refinery_coverage['crude_input_total_bbl'] <= 0,
    ],
    [
        'missing from RRC extract',
        'in extract, no Crude Oil row',
        'Crude Oil row present, zero crude charge',
    ],
    default='ok',
)

missing_crude = refinery_coverage[refinery_coverage['status'] != 'ok'].sort_values(['district', 'id'])
chartable_ids = set(refinery_coverage.loc[refinery_coverage['status'] == 'ok', 'id'])

print(f'Refineries with crude intake series: {rrc_crude_by_refinery["id"].nunique()}')
print(f'Mapped refineries flagged (no usable crude filings): {len(missing_crude)}')
print(f'Period: {rrc_crude_by_refinery["period"].min()} → {rrc_crude_by_refinery["period"].max()}')

if len(missing_crude):
    display(missing_crude)
else:
    print('All mapped refineries have crude intake filings.')

display(
    rrc_crude_by_refinery.sort_values(['plant_name', 'period'])
    .head(12)[['district', 'id', 'plant_name', 'period', 'input_bbl', 'kbd']]
)

Refineries with crude intake series: 28
Mapped refineries flagged (no usable crude filings): 4
Period: 2020-01 → 2025-12


,id,ownser,plant_name,district,in_rrc_extract,has_crude_row,crude_input_total_bbl,status
16,03-0253,Marathon,Texas City,Texas Gulf Coast,False,False,0.0,missing from RRC extract
23,Apr-22,POTAK,Pin Oak Corpus,Texas Gulf Coast,False,False,0.0,missing from RRC extract
25,Apr-69,Buckeye Texas Processing,Buckeye Texas Processing,Texas Gulf Coast,True,True,0.0,"Crude Oil row present, zero crude charge"
1,01-0717,Calumet Shreveport Fuels LLC,Calumet San Antonia Refinery,Texas Inland,False,False,0.0,missing from RRC extract


,district,id,plant_name,period,input_bbl,kbd
135,Texas Gulf Coast,03-0012,Baytown refinery,2020-01,16665222.0,537.587806
136,Texas Gulf Coast,03-0012,Baytown refinery,2020-02,16010456.0,552.084690
137,Texas Gulf Coast,03-0012,Baytown refinery,2020-03,17140001.0,552.903258
138,Texas Gulf Coast,03-0012,Baytown refinery,2020-04,14665269.0,488.842300
139,Texas Gulf Coast,03-0012,Baytown refinery,2020-05,15080146.0,486.456323
140,Texas Gulf Coast,03-0012,Baytown refinery,2020-06,15451528.0,515.050933
141,Texas Gulf Coast,03-0012,Baytown refinery,2020-07,16223515.0,523.339194
142,Texas Gulf Coast,03-0012,Baytown refinery,2020-08,15759901.0,508.383903
143,Texas Gulf Coast,03-0012,Baytown refinery,2020-09,15083936.0,502.797867
144,Texas Gulf Coast,03-0012,Baytown refinery,2020-10,14426602.0,465.374258


In [249]:
refinery_coverage

,id,ownser,plant_name,district,in_rrc_extract,has_crude_row,crude_input_total_bbl,status
0,01-0692,Calumet San Antonio Refining,The San Antonio Refinery,Texas Inland,True,True,3.953499e+07,ok
1,01-0717,Calumet Shreveport Fuels LLC,Calumet San Antonia Refinery,Texas Inland,False,False,0.000000e+00,missing from RRC extract
2,01-0763,Lazarus Energy Holdings,Nixon Refinery,Texas Inland,True,True,2.464505e+07,ok
3,02-0057,Diamond Shamrock,Three Rivers Refinery,Texas Inland,True,True,1.721805e+08,ok
4,03-0006,Houston Refining,Houston Refining,Texas Gulf Coast,True,True,4.351028e+08,ok
5,03-0008,Pasadena Refining System,Pasadena Refining System,Texas Gulf Coast,True,True,1.762672e+08,ok
6,03-0012,Exxon Baytown,Baytown refinery,Texas Gulf Coast,True,True,1.143738e+09,ok
7,03-0014,Deer Park Refining Limited,Deer Park refinery,Texas Gulf Coast,True,True,5.606734e+08,ok
8,03-0024,Premcor Refining Group,Premcor Port Arthur Refinery,Texas Gulf Coast,True,True,6.355615e+08,ok
9,03-0031,Valero Houston Refining,Valero Houston Refinery,Texas Gulf Coast,True,True,4.478912e+08,ok


In [250]:
# ── Crude intake time series by refinery (dropdown) ───────────────────────────
def refinery_crude_kbd_dropdown(df, coverage, chartable_ids):
    """Monthly kbd line + period-average line; average toggles via legend click."""
    plot_df = df[df['id'].isin(chartable_ids) & (df['kbd'] > 0)].copy()
    if plot_df.empty:
        print('No refineries with positive crude intake to chart.')
        return go.Figure()

    ref_stats = (
        plot_df.groupby(['id', 'iir_map', 'district'], as_index=False)
        .agg(avg_kbd=('kbd', 'mean'))
        .sort_values('avg_kbd', ascending=False)
    )

    x_span = [plot_df['period_dt'].min(), plot_df['period_dt'].max()]

    def _series(ref_id):
        return plot_df[plot_df['id'] == ref_id].sort_values('period_dt')

    default = ref_stats.iloc[0]
    default_sub = _series(default['id'])
    default_label = f"{default['iir_map'] or default['id']} | {default['district']}"
    default_avg = default['avg_kbd']

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=default_sub['period_dt'],
        y=default_sub['kbd'],
        mode='lines',
        name='Crude intake',
        line=dict(color='#2a6f7a', width=2),
    ))
    fig.add_trace(go.Scatter(
        x=x_span,
        y=[default_avg, default_avg],
        mode='lines',
        name=f'Average ({default_avg:.0f} kbd)',
        line=dict(color='#c44e52', width=1.5, dash='dash'),
    ))

    buttons = []
    for _, row in ref_stats.iterrows():
        sub = _series(row['id'])
        label = f"{row['iir_map'] or row['id']} | {row['district']}"
        avg_kbd = row['avg_kbd']
        buttons.append(dict(
            label=label,
            method='update',
            args=[
                {
                    'x': [sub['period_dt'], x_span],
                    'y': [sub['kbd'], [avg_kbd, avg_kbd]],
                    'name': ['Crude intake', f'Average ({avg_kbd:.0f} kbd)'],
                },
                {'title': f'Crude intake — {label}'},
            ],
        ))

    fig.update_layout(
        title=f'Crude intake — {default_label}',
        xaxis=dict(
            tickformat='%d/%m/%Y',
            dtick='M3',
            tickangle=-90,
            showgrid=False,
        ),
        yaxis=dict(title='kbd', gridcolor='#e0e0e0', zeroline=False),
        hovermode='x unified',
        height=480,
        plot_bgcolor='white',
        margin=dict(t=90, b=120),
        #legend=dict(title='Click label to toggle'),
        updatemenus=[{
            'buttons': buttons,
            'direction': 'down',
            'showactive': True,
            'x': 0.0,
            'y': 1.18,
            'xanchor': 'left',
            'yanchor': 'top',
        }],
    )
    return fig


fig_crude_kbd = refinery_crude_kbd_dropdown(rrc_crude_by_refinery, refinery_coverage, chartable_ids)
fig_crude_kbd.show()


In [251]:
# ── Coverage: refineries in mapping but missing from RRC data ────────────────
ids_in_data = set(rrc['id'].unique())
ids_in_mapping = set(mapping['id'].unique())
missing_ids = sorted(ids_in_mapping - ids_in_data)

missing_tbl = mapping[mapping['id'].isin(missing_ids)][['id', 'ownser', 'plant_name', 'district']]
print(f'Refineries in mapping but not in RRC extract: {len(missing_tbl)}')
missing_tbl


Refineries in mapping but not in RRC extract: 3


,id,ownser,plant_name,district
1,01-0717,Calumet Shreveport Fuels LLC,Calumet San Antonia Refinery,Texas Inland
16,03-0253,Marathon,Texas City,Texas Gulf Coast
23,Apr-22,POTAK,Pin Oak Corpus,Texas Gulf Coast


In [252]:
# ── Refinery contribution to district total (latest month) ─────────────────
latest_period = rrc_prod['period'].max()
contrib = (
    rrc_by_refinery[rrc_by_refinery['period'] == latest_period]
    .groupby(['district', 'id', 'plant_name'], as_index=False)['mbbl']
    .sum()
)
contrib['share_pct'] = 100 * contrib['mbbl'] / contrib.groupby('district')['mbbl'].transform('sum')

for district in contrib['district'].unique():
    print(f'\n{district} — top refineries ({latest_period}):')
    display(contrib[contrib['district'] == district].sort_values('mbbl', ascending=False).head(8))



Texas Gulf Coast — top refineries (2025-12):


,district,id,plant_name,mbbl,share_pct
1,Texas Gulf Coast,03-0012,Baytown refinery,24130.260,13.242417
9,Texas Gulf Coast,03-0203,port arthur refinery,23960.238,13.149110
7,Texas Gulf Coast,03-0138,Beaumont Refinery,23450.945,12.869616
5,Texas Gulf Coast,03-0052,Galveston Bay Refinery,21238.672,11.655545
15,Texas Gulf Coast,Apr-18,Corpus Christi,12538.556,6.881019
3,Texas Gulf Coast,03-0024,Premcor Port Arthur Refinery,11262.770,6.180882
10,Texas Gulf Coast,03-0228,Sweeny refinery,10936.067,6.001591
2,Texas Gulf Coast,03-0014,Deer Park refinery,9155.446,5.024406



Texas Inland — top refineries (2025-12):


,district,id,plant_name,mbbl,share_pct
25,Texas Inland,10-0026,McKee refinery,5491.750,22.763827
24,Texas Inland,10-0013,Borger refinery,4998.285,20.718367
23,Texas Inland,08-0005,El Paso Refinery,4541.424,18.824635
21,Texas Inland,02-0057,Three Rivers Refinery,3169.500,13.137879
22,Texas Inland,06-0111,Tyler Refinery,2681.965,11.117000
26,Texas Inland,Aug-69,Big Spring,2437.979,10.105655
19,Texas Inland,01-0692,The San Antonio Refinery,527.470,2.186413
20,Texas Inland,01-0763,Nixon Refinery,276.525,1.146222


In [253]:
# ── refp2 vs refp: quick verdict ─────────────────────────────────────────────
# Rank routes by absolute median % difference across districts.
verdict = (
    compare.groupby('route')
    .agg(
        median_pct_diff=('pct_diff', 'median'),
        mean_abs_pct_diff=('pct_diff', lambda s: np.abs(s).median()),
        corr=('rrc_mbbl', lambda s: s.corr(compare.loc[s.index, 'eia_mbbl'])),
    )
    .sort_values('mean_abs_pct_diff')
)
print('Production total (EP00) — which EIA route is closer to RRC overall?')
verdict

best_route = verdict.index[0]
print(f'\nCloser route on median |% diff|: {best_route}')
print('Note: refp2 = refinery only; refp = refinery + blender. Small gaps between them')
print('suggest blender activity is not the main driver of RRC/EIA differences.')


Production total (EP00) — which EIA route is closer to RRC overall?

Closer route on median |% diff|: refp
Note: refp2 = refinery only; refp = refinery + blender. Small gaps between them
suggest blender activity is not the main driver of RRC/EIA differences.
